1- Importing Libraries
In this section, we import the necessary libraries for data manipulation (Pandas, Numpy), data visualization (Matplotlib, Seaborn), and machine learning (Scikit-Learn, XGBoost).

In [ ]:
# Library for data handling
import pandas as pd
# Library for mathematical operations and matrices
import numpy as np
# To divide the data into training and testing
from sklearn.model_selection import train_test_split
# To unify values
from sklearn.preprocessing import StandardScaler
# Support Vector Machine Algorithms
from sklearn.svm import SVC
# Model Evaluation Criteria
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
# XGBoost classification algorithm
from xgboost import XGBClassifier
# Importing classification algorithms: SVM and XGBoost
from sklearn.svm import SVC, LinearSVC
from xgboost import XGBClassifier

2- Data Loading and Merging
We load both red and white wine datasets. A new column `wine_type` is added to distinguish between them (0 for red, 1 for white), then we combine them into a single dataset for analysis.

In [ ]:
# Reading Red Wine Data
red = pd.read_csv("data/winequality-red.csv", sep=';')
#Read white wine data
white = pd.read_csv("data/winequality-white.csv", sep=';')
# Add a column specifying the type of red wine = 0
red['wine_type'] = 0
# Add a column specifying the type of wine white = 1
white['wine_type'] =1
# Merging red and white wine data into a single dataset1
data = pd.concat([red, white], axis=0)

3- Data Preprocessing
To convert this into a classification problem, we transform the quality column into a binary target:
- 1 (Good): if quality > 5
- 0 (Bad): if quality <= 5

In [ ]:
# Converting quality value to binary classification
data['quality'] = data['quality'].apply(lambda x: 1 if x > 5 else 0)

4- Data Splitting
We separate the features (X) from the target (y) and split the data into a Training set (80%) and a Testing set (20%)to evaluate our models performance

In [ ]:
# Separating the characteristics from the target group
X = data.drop('quality', axis=1)
# Dividing the data into a training set (80%) and a test set (20%)
y = data['quality']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

5- Feature Scaling
Since SVM is sensitive to the scale of data, we use StandardScaler to normalize the features, ensuring they all have a mean of 0 and a standard deviation of 1

In [ ]:
# Creating an object to standardize values
scaler = StandardScaler()
# Applying standardization to training data
X_train = scaler.fit_transform(X_train)
# Applying the same standardization to test data
X_test = scaler.transform(X_test)

6- Model Training and Evaluation
We will train three different classifiers as required:

1-Hard Margin SVM:Using a very high C parameter to minimize margin violations

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Create a Hard Margin SVM model using LinearSVC (quick)
svm_hard = LinearSVC(C=1e6, max_iter=5000)

# Train the model on the training data
svm_hard.fit(X_train, y_train)

# Predict the test data classes
y_pred_hard = svm_hard.predict(X_test)

# Calculate the model accuracy
acc_hard = accuracy_score(y_test, y_pred_hard)

# Calculate the F1-Score
f1_hard = f1_score(y_test, y_pred_hard)

# Calculate the ROC-AUC
auc_hard = roc_auc_score(y_test, svm_hard.decision_function(X_test))

# Print the Hard Margin SVM results
print("Hard Margin SVM")
print(acc_hard, f1_hard, auc_hard)

2-Soft Margin SVM: Using a standard C parameter to allow some

In [ ]:
# Create a Soft Margin SVM model using a linear kernel
svm_soft = SVC(kernel='linear', C=1, probability=True)

# Train the model on the training data
svm_soft.fit(X_train, y_train)

# Predict the test data classes
y_pred = svm_soft.predict(X_test)

# Calculate the model accuracy
acc = accuracy_score(y_test, y_pred)

# Calculate the F1-Score
f1 = f1_score(y_test, y_pred)

# Calculate the ROC-AUC (using the predicted values)
auc = roc_auc_score(y_test, svm_soft.predict_proba(X_test)[:, 1])

# Print the results
print(acc, f1, auc)

acc_soft = acc


3-XGBoost: A powerful gradient boosting algorithm.

In [ ]:
# Create an XGBoost classification model
xgb = XGBClassifier(eval_metric='logloss', random_state=42)
# Train the model on the training data
xgb.fit(X_train, y_train)
# Predict the classes of the test data
y_pred = xgb.predict(X_test)
# Calculate the model accuracy
acc = accuracy_score(y_test, y_pred)
# Calculate the F1-Score
f1 = f1_score(y_test, y_pred)
# Calculate the ROC-AUC
auc = roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1])
# Print the results
print(acc, f1, auc)
acc_xgb = acc


7- Visual Comparison of Results
In this section, we visualize the performance of the three models using a bar chart to compare their Accuracy scores.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Preparing data for drawing
model_names = ['Hard Margin SVM', 'Soft Margin SVM', 'XGBoost']
accuracy_scores = [acc_hard, acc_soft, acc_xgb]
# Setting up the graph format
plt.figure(figsize=(10, 6))
# Draw columns
bars = plt.bar(model_names, accuracy_scores, color=['#4C72B0', '#55A868', '#C44E52'])
# Add drawing details
plt.title('Comparison of Model Accuracy', fontsize=15)
plt.xlabel('Classifier', fontsize=12)
plt.ylabel('Accuracy Score', fontsize=12)
plt.ylim(0, 1.0)
# Add numbers above each column for clarity
for bar in bars:
 yval = bar.get_height()
 plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, round(yval, 3), ha='center', va='bottom')

plt.show()